In [3]:
import pandas as pd
import numpy as np
import sklearn

In [4]:
def var_engineering(df):
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #RECODING CERTAIN VARS
    #~~~~~~~~~~~~~~~~~~~~~~

    #gender
    mapping = {'female': 1, 'male': 0}
    df['sex_dum'] = df['Sex'].map(mapping)

    #Embarked
    mapping = {'S': 1, 'C': 2, 'Q': 0}
    df['Embark_dum'] = df['Embarked'].map(mapping)

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #VAR CONSTRUCTION / ENGINGEERING

    #~~~~~~~~~~~~~~~~~~~~~~
    #log fare

    df['Fare'] = df['Fare'].fillna(df['Fare'].median())
    df['ln_fare'] = np.log1p(df['Fare']) #makes a var called ln_fare using numpy ln() func

    #~~~
    #filling missing age values
    df['Age'] = df['Age'].fillna(df['Age'].median())

    #~~~~~~~~~~~~~~~~~~~~~~
    #age^2
    df['sq_age'] = df['Age'] * df['Age']

    #~~~~~~~~~~~~~~~~~~~~~~
    #making dummies for each pclass
    for class_var in [1, 2]: #the omitted var will be pclass = 3
        #this loops through the specified variable and makes a new column with a dummy of that var
        df[f'pclass_{class_var}'] = (df['Pclass'] == class_var).astype(int)
        
        #we can loop through to make an interaction variable as well
        df[f'sexXpc_{class_var}'] = (df['Pclass'] == class_var).astype(int) * df['sex_dum']

    return df

#~~~~~~~~~~~~~~~~~~~~~~
#Selecting features (X / regressors)
feat = ['pclass_1', 'pclass_2', 'sex_dum', 'sexXpc_1', 'sexXpc_2', 'Age', 'sq_age', 'SibSp', 'Parch', 'ln_fare']

df = var_engineering(pd.read_csv("/kaggle/input/competitions/titanic/train.csv"))
df_test = var_engineering(pd.read_csv("/kaggle/input/competitions/titanic/test.csv"))

In [13]:
train_y = df.Survived
train_X = df[feat]
test_X = df_test[feat]

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import mean_absolute_error

train_X, val_X, train_y, val_y = train_test_split(df[feat], df.Survived, test_size=0.2, random_state=1)

df_model = DecisionTreeClassifier(max_leaf_nodes = 25, random_state=1)
df_model.fit(train_X, train_y)

from sklearn.metrics import accuracy_score
print("Validation accuracy:", accuracy_score(val_y, df_model.predict(val_X)))

Validation accuracy: 0.8212290502793296


In [14]:
df_model = DecisionTreeClassifier(max_leaf_nodes = 10, random_state = 1)

df_model = df_model.fit(train_X, train_y, sample_weight=None, check_input=True)

df_predictions = df_model.predict(test_X)

output = pd.DataFrame({'PassengerId': df_test.PassengerId, 'Survived': df_predictions})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!
